# Sessão 6 · Prática guiada: explorar as vendas de 2024 e 2025

A direção da Morabeza ainda não quer um painel: quer respostas rápidas sobre dois anos de vendas.
Vais responder-lhes passo a passo, com NumPy e pandas.

**Como trabalhar**

1. Corre as células por ordem, de cima para baixo (Shift + Enter).
2. Onde vires `...`, substitui pelo teu código.
3. Depois de cada resposta há uma célula `confere(...)`. Diz-te se está certo, sem mostrar a solução.

Se uma célula der erro, lê a última linha da mensagem: é quase sempre aí que está a pista.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Procura a pasta com os CSV: no Codespace (data/tabelas) ou ao lado do notebook (Colab)
candidatas = [Path("data/tabelas"), Path("../data/tabelas"), Path(".")]
PASTA = next(p for p in candidatas if (p / "vendas.csv").exists())
print("Dados em:", PASTA.resolve())

Dados em: /workspaces/morabeza_dados/data/tabelas


In [2]:
import hashlib

def confere(valor, codigo):
    """Diz se a tua resposta está certa, sem mostrar a solução."""
    if valor is ...:
        print("Ainda não respondeste a este passo.")
        return
    if hasattr(valor, "sum"):   # um array ou uma coluna: confere a soma
        valor = valor.sum()
    try:
        texto = str(round(float(valor), 1))
    except (TypeError, ValueError):
        texto = str(valor)
    certo = hashlib.md5(texto.encode()).hexdigest()[:6] == codigo
    print("Certo!" if certo else "Ainda não. Revê o passo e volta a correr a célula.")

## Parte 1 · NumPy (10 min)

Quatro vendas pequenas, para aquecer. Primeiro com arrays, sem pandas.

In [3]:
q = np.array([5, 25, 12, 40])          # quantidades
preco = np.array([100, 50, 1000, 90])  # preço unitário
q, preco

(array([ 5, 25, 12, 40]), array([ 100,   50, 1000,   90]))

**1.1** Calcula o total de cada venda (quantidade vezes preço), sem `for`. Guarda em `totais`.

In [4]:
totais = q*preco
totais

array([  500,  1250, 12000,  3600])

In [5]:
confere(totais, "9a5db8")

Certo!


**1.2** Cria a máscara das vendas com quantidade igual ou superior a 20 e conta quantas são.

In [6]:
grandes = q >= 20          # array de True e False
n_grandes = (q >= 20).sum()        # quantas são True
grandes, n_grandes

(array([False,  True, False,  True]), np.int64(2))

In [7]:
confere(n_grandes, "d1bd83")

Certo!


**1.3** Com `np.where`, aplica 5% de desconto às vendas grandes (quantidade de 20 ou mais).
Guarda em `com_desconto` o total de cada venda já com desconto.

In [8]:
desconto = np.where(q >= 20, 0.05, 0.0)
com_desconto = totais * (1-desconto)

In [9]:
confere(com_desconto, "8a4f25")

Certo!


## Parte 2 · Ler e conhecer os dados (10 min)

Agora as tabelas a sério. A célula seguinte lê as três tabelas. Repara no `parse_dates`.

In [10]:
vendas = pd.read_csv(PASTA / "vendas.csv", parse_dates=["data"])
clientes = pd.read_csv(PASTA / "clientes.csv", parse_dates=["data_registo"])
produtos = pd.read_csv(PASTA / "produtos.csv")
vendas.head()

,venda_id,data,cliente_id,produto_id,quantidade,preco_unitario
0,V000612,2024-01-01,C0007,P017,18,180
1,V001503,2024-01-01,C0017,P027,9,690
2,V001689,2024-01-01,C0019,P028,7,150
3,V001868,2024-01-01,C0021,P008,17,260
4,V002287,2024-01-01,C0030,P012,20,130


**2.1** Usa `info()` para ver as colunas e os tipos das vendas. Que tipo tem a coluna `data`? Há valores em falta?

In [11]:
vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 33788 entries, 0 to 33787
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   venda_id        33788 non-null  str           
 1   data            33788 non-null  datetime64[us]
 2   cliente_id      33788 non-null  str           
 3   produto_id      33788 non-null  str           
 4   quantidade      33788 non-null  int64         
 5   preco_unitario  33788 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(3)
memory usage: 1.5 MB


In [12]:
...

Ellipsis

**2.2** Quantas linhas tem a tabela de vendas? (usa `shape`)

In [13]:
n_vendas = vendas.shape[0]
n_vendas

33788

In [14]:
confere(n_vendas, "85b6dd")

Certo!


**2.3** Usa `describe()` nas vendas. Qual é a quantidade média por venda? E o preço máximo?

In [15]:
vendas.describe()

,data,quantidade,preco_unitario
count,33788,33788.000000,33788.000000
mean,2025-01-04 20:55:37.895110,16.859122,300.757074
min,2024-01-01 00:00:00,1.000000,60.000000
25%,2024-07-25 00:00:00,8.000000,150.000000
50%,2025-01-05 00:00:00,16.000000,210.000000
75%,2025-06-18 00:00:00,24.000000,390.000000
max,2025-12-31 00:00:00,39.000000,1250.000000
std,NaN,10.389558,258.358185


**2.4** Quantos clientes há em Santiago? Usa `value_counts()` na coluna `ilha` dos clientes.

In [16]:
contagem_ilhas = clientes["ilha"].value_counts()
contagem_ilhas

ilha
Santiago       205
São Vicente     75
Sal             74
Boa Vista       44
Santo Antão     41
Fogo            41
Name: count, dtype: int64

In [17]:
n_santiago = clientes["ilha"].value_counts()["Santiago"]

In [18]:
confere(n_santiago, "aba3bc")

Certo!


## Parte 3 · Uma coluna nova (5 min)

A tabela não tem o total de cada venda. Cria-o, vetorizado, como fizeste com os arrays.

**3.1** Cria a coluna `total` (quantidade vezes preço unitário) e calcula a faturação dos dois anos.

In [19]:
vendas["total"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas[["quantidade", "preco_unitario", "total"]].head()
faturacao = vendas["total"].sum()
faturacao

np.int64(171752890)

In [20]:
confere(faturacao, "c522a0")

Certo!


## Parte 4 · Filtrar (10 min)

Lembra-te: com pandas usa-se `&` e `|`, e cada condição vai entre parênteses.

**4.1** Quantas vendas têm quantidade igual ou superior a 20?

In [21]:
n_grandes = (vendas["quantidade"] >= 20).sum()
n_grandes

np.int64(12979)

In [22]:
confere(n_grandes, "6cee6d")

Certo!


**4.2** Quanto se faturou em 2025? Pista: `vendas["data"].dt.year` dá o ano de cada venda.

In [23]:
fat_2025 = vendas.loc[vendas["data"].dt.year == 2025, "total"].sum()
fat_2025

np.int64(86115500)

In [24]:
confere(fat_2025, "400ab7")

Certo!


**4.3** Quantos clientes do tipo HORECA há no Sal? Usa duas condições com `&`.

In [25]:
n_horeca_sal = ((clientes["tipo"] == "HORECA") & (clientes["ilha"] == "Sal")).sum()
n_horeca_sal

np.int64(24)

In [26]:
confere(n_horeca_sal, "e23763")

Certo!


**4.4** Quantos clientes são do Barlavento? Para estes dados, conta São Vicente, Sal, Santo Antão e Boa Vista. Usa `isin`.

In [27]:
barlavento = ["São Vicente", "Sal", "Santo Antão", "Boa Vista"]
n_barlavento = clientes["ilha"].isin(barlavento).sum()
n_barlavento

np.int64(234)

In [28]:
confere(n_barlavento, "ad3b03")

Certo!


## Parte 5 · Agrupar (10 min)

`groupby` divide as linhas por grupo, faz uma conta a cada grupo e junta os resultados.

**5.1** Faturação por ano. Compara 2024 com 2025: a empresa cresceu?

In [29]:
por_ano = vendas.groupby(vendas["data"].dt.year)["total"].sum()
por_ano

fat_2024 = por_ano.loc[2024]
fat_2025 = por_ano.loc[2025]

diferenca = fat_2025 - fat_2024
variacao_pct = (diferenca / fat_2024) * 100
cresceu = fat_2025 > fat_2024

print(f"2024: {fat_2024}")
print(f"2025: {fat_2025}")
print(f"Diferença: {diferenca} ({variacao_pct:.2f}%)")
print("Cresceu?", cresceu)

2024: 85637390
2025: 86115500
Diferença: 478110 (0.56%)
Cresceu? True


Quanto se faturou em 2024? Tira o valor de `por_ano` com `.loc`.

In [30]:
fat_2024 = por_ano.loc[2024]
fat_2024

np.int64(85637390)

In [31]:
confere(fat_2024, "7e7559")

Certo!


**5.2** Os 5 produtos que mais faturaram. Agrupa por `produto_id`, soma o `total` e ordena do maior para o menor.

In [32]:
top5 = (
    vendas.groupby("produto_id")["total"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
top5

produto_id
P024    23057500
P005    16169400
P027    12509700
P022    11494600
P030     9064800
Name: total, dtype: int64

Qual é o código do produto que mais faturou? É o primeiro rótulo do índice de `top5`.

In [33]:
primeiro = top5.index[0]
primeiro

'P024'

In [34]:
confere(primeiro, "3e2f46")

Certo!


**5.3** Quantos clientes há de cada tipo em cada ilha? Agrupa os clientes por `ilha` e `tipo` e usa `size()`.

In [35]:
clientes_ilha_tipo = clientes.groupby(["ilha", "tipo"]).size()
clientes_ilha_tipo

ilha         tipo   
Boa Vista    Empresa      5
             HORECA      19
             Retalho     20
Fogo         Empresa      9
             HORECA      15
             Retalho     17
Sal          Empresa     15
             HORECA      24
             Retalho     35
Santiago     Empresa     29
             HORECA      66
             Retalho    110
Santo Antão  Empresa      8
             HORECA      12
             Retalho     21
São Vicente  Empresa     18
             HORECA      26
             Retalho     31
dtype: int64

## Parte 6 · Desafio (5 min)

A direção pondera dar 5% de desconto em todas as vendas com quantidade de 20 ou mais.
Quanto custaria esse desconto nos dois anos? Usa `np.where` sobre as colunas do DataFrame.

In [37]:
vendas["total_desconto"] = np.where(vendas["quantidade"] >= 20, vendas["total"] * 0.05, 0)
custo = vendas["total_desconto"].sum()
custo

np.float64(5469452.5)

In [38]:
confere(custo, "000797")

Certo!


## Se acabares cedo

**E.1** Qual foi o mês com maior faturação? Pista: `vendas["data"].dt.to_period("M")` dá o mês de cada venda, e `idxmax()` devolve o rótulo do maior valor.

In [39]:
melhor_mes = vendas.groupby(vendas["data"].dt.to_period("M"))["total"].sum().idxmax()
melhor_mes

Period('2024-12', 'M')

In [40]:
confere(melhor_mes, "34c644")

Certo!


**E.2** Os códigos do top 5 dizem pouco à direção. Mostra os nomes dos produtos.
Pista: `produtos.set_index("produto_id")` e depois `.loc[top5.index, "nome"]`.

In [ ]:
nomes_top5 = produtos.set_index("produto_id").loc[top5.index, "nome"]
nomes_top5

##top5_com_nome = top5.to_frame(name="faturacao").join(
##    produtos.set_index("produto_id")["nome"]
##)
##top5_com_nome

produto_id
P024         Detergente roupa 3 kg
P005                  Grogue 70 cl
P027    Papel higiénico (12 rolos)
P022         Queijo de cabra 250 g
P030           Gel de banho 750 ml
Name: nome, dtype: str

Na próxima sessão juntas estas tabelas com `merge`, mudas a forma com `pivot_table` e montas o painel de vendas no Laboratório 4.